# Chapter 3.3 EB Ablations

**TL;DR** — Starts from the EB main model cache written by Chapter 3.2 and runs the Chapter 3 diagnostic suite. The reader finishes with solver, CNF-style training-cost, time-sampling, capacity, straightness, and paper-table artifacts under `figures/ch03` and `tables/ch03`.

**Prerequisites** - Chapter 3.2 (EB main CFM model cache). Run chapter3_2 first to generate the cached model.

**Outputs**
- `figures/ch03/fig03_10_nfe_vs_endpoint_error.png`
- `figures/ch03/fig03_10_nfe_vs_endpoint_error.pdf`
- `figures/ch03/figE1_fm_vs_cnf_compute_quality.png`
- `figures/ch03/figE1_fm_vs_cnf_compute_quality.pdf`
- `figures/ch03/figE1_fm_vs_cnf_cumulative_time.png`
- `figures/ch03/figE1_fm_vs_cnf_cumulative_time.pdf`
- `figures/ch03/figE3_time_sampling_distributions.png`
- `figures/ch03/figE3_time_sampling_distributions.pdf`
- `figures/ch03/figE3_time_sampling_val_mse.png`
- `figures/ch03/figE3_time_sampling_val_mse.pdf`
- `figures/ch03/figE3_time_sampling_endpoint_mmd.png`
- `figures/ch03/figE3_time_sampling_endpoint_mmd.pdf`
- `figures/ch03/figE3_time_sampling_final_bar.png`
- `figures/ch03/figE3_time_sampling_final_bar.pdf`
- `figures/ch03/figE2_capacity_val_mse.png`
- `figures/ch03/figE2_capacity_val_mse.pdf`
- `figures/ch03/figE2_capacity_endpoint_mmd.png`
- `figures/ch03/figE2_capacity_endpoint_mmd.pdf`
- `figures/ch03/figE5_straightness_hist.png`
- `figures/ch03/figE5_straightness_hist.pdf`
- `figures/ch03/figE5_endpoint_distance_vs_straightness.png`
- `figures/ch03/figE5_endpoint_distance_vs_straightness.pdf`
- `figures/ch03/figE5_representative_trajectories_phate.png`
- `figures/ch03/figE5_representative_trajectories_phate.pdf`
- `tables/ch03/table03_01_solver_diagnostics.csv`
- `tables/ch03/tableE1_cfm_vs_cnf_endpoint.csv`
- `tables/ch03/tableE3_time_sampling_ablation.csv`
- `tables/ch03/tableT2_training_hyperparams_capacity.csv`
- `tables/ch03/tableE5_trajectory_straightness.csv`
- `tables/ch03/paper_table03_01_solver_diagnostics.csv`
- `tables/ch03/paper_table03_01_solver_diagnostics.md`
- `tables/ch03/paper_table03_01_solver_diagnostics.tex`
- `tables/ch03/paper_tableE1_cfm_vs_cnf_endpoint.csv`
- `tables/ch03/paper_tableE1_cfm_vs_cnf_endpoint.md`
- `tables/ch03/paper_tableE1_cfm_vs_cnf_endpoint.tex`
- `tables/ch03/paper_tableE3_time_sampling_ablation.csv`
- `tables/ch03/paper_tableE3_time_sampling_ablation.md`
- `tables/ch03/paper_tableE3_time_sampling_ablation.tex`
- `tables/ch03/paper_tableT2_training_hyperparams_capacity.csv`
- `tables/ch03/paper_tableT2_training_hyperparams_capacity.md`
- `tables/ch03/paper_tableT2_training_hyperparams_capacity.tex`
- `tables/ch03/paper_tableE5_trajectory_straightness_summary.csv`
- `tables/ch03/paper_tableE5_trajectory_straightness_summary.md`
- `tables/ch03/paper_tableE5_trajectory_straightness_summary.tex`

**Runtime** - `QUICK_MODE` defaults on through the Chapter 3 config and uses the cached main model to keep ablations tutorial-sized; `SMOKE_MODE` further shrinks repeated training and sampling for CI. On CPU, this is one of the longer notebooks because it runs several diagnostic trainings, so smoke mode is the practical baseline for fast checks.

**Key parameters**
- `SEED` defaults to `42` via `CH03_SEED` and selects the Chapter 3.2 cache.
- `QUICK_MODE` defaults to `CH03_QUICK=1`; `SMOKE_MODE` uses `CH03_SMOKE_MODE=1`.
- State and cost space: standardized EB `PC-20`; PHATE is display-only.
- Solver diagnostic varies NFE/tolerance and reports endpoint error.
- Time-sampling, capacity, and CNF-style diagnostics reuse Chapter 3 training-step and batch-size knobs.
- Straightness diagnostics sample validation trajectories and summarize endpoint distance versus path ratio.

## Tutorial setup

The setup fixes seeds, resolves project paths, imports the shared reporting helpers, and defines one local `save_and_show` wrapper so each figure is written and displayed in a single code cell.


In [ ]:
from pathlib import Path
import json
import os
import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

try:
    import torch
except ImportError as exc:
    raise ImportError("This notebook requires PyTorch for CFM training and sampling.") from exc

import sys

ROOT_HINT = Path.cwd().resolve()
if not (ROOT_HINT / "src").is_dir():
    ROOT_HINT = ROOT_HINT.parent
if str(ROOT_HINT) not in sys.path:
    sys.path.insert(0, str(ROOT_HINT))

from src.tutorial_init import apply_tutorial_plot_style, bootstrap, make_ch03_run_config, make_save_and_show

SEED = int(os.environ.get("CH03_SEED", "42"))
QUICK_MODE = os.environ.get("CH03_QUICK", "1") == "1"
boot = bootstrap(chapter="ch03", seed=SEED, quick_mode=QUICK_MODE)
PROJECT_ROOT = boot.project_root
FIG_DIR = boot.fig_dir
OUT_DIR = boot.out_dir
DEVICE = boot.device

from src.visualization import flow_matching as ch03
from src.core import ot as fm_ot
from src.utils import set_seed
from src.data.loading import load_eb_timecourse_for_ch03, copy_trajectorynet_eb_to_data
from src.core.models import VelocityMLP, count_parameters
from src.core.losses import cfm_loss_from_pairs
from src.core.train import train_cfm_steps
from src.core.sampling import euler_sample, midpoint_sample, odeint_sample
from src.evaluation.metrics import endpoint_metrics, mmd_rbf, sliced_wasserstein_distance, trajectory_straightness


In [ ]:
config = make_ch03_run_config()
SEED = config.seed
QUICK_MODE = config.quick_mode
SMOKE_MODE = config.smoke_mode
PAPER_FIGURE_MODE = config.paper_figure_mode
DEVICE = config.device
rng = np.random.default_rng(SEED)

FULL_RUN_HINTS = {
    "eb_train_steps": 10000,
    "cnf_steps": 600,
    "time_ablation_steps": 1200,
    "capacity_steps": 1200,
    "notes": "Set CH03_QUICK=0 for full-run settings; default remains quick and reproducible.",
}

context = ch03.make_ch03_context(PROJECT_ROOT)
FIG_DIR = context.fig_dir
TABLE_DIR = context.table_dir
OUT_DIR = context.output_dir
PAPER_COLORS = ch03.PAPER_COLORS

tracker = ch03.Ch03ArtifactTracker(context, paper_figure_mode=PAPER_FIGURE_MODE)
skipped_items = []

apply_tutorial_plot_style()
save_and_show = make_save_and_show(FIG_DIR, write_pdf=PAPER_FIGURE_MODE, save_fn=tracker.save_figure)

set_paper_style = apply_tutorial_plot_style
add_panel_label = ch03.add_panel_label
short_strategy_label = ch03.short_strategy_label
clean_spines = ch03.clean_spines
format_metric_axis = ch03.format_metric_axis
add_note = ch03.add_note
as_np = ch03.as_np
subsample_idx = ch03.subsample_idx
robust_limits = ch03.robust_limits
format_axis = ch03.format_axis

print(f"device={DEVICE}; quick_mode={QUICK_MODE}; smoke_mode={SMOKE_MODE}; paper_figure_mode={PAPER_FIGURE_MODE}; seed={SEED}")


## 1. Load EB inputs and the main-model cache

The ablations reuse the EB split, PHATE display projector, and cached main EB CFM model from Chapter 3.2. Metrics remain in the 20D PC training space; PHATE is only used for display panels.


### Hypothesis
The ablations assume the Chapter 3.2 EB split, cached main CFM vector field, and display projector are the fixed reference for all diagnostics.

### Setup
The setup loads `eb`, `X_source_20d`, `X_target_20d`, `pair_batch_fn`, and the cached `eb_model` from `MAIN_MODEL_PATH`. Metrics use held-out PC-space arrays such as `x0_eval_20d` and `target_eval_20d`, while `phate_projector` is kept for display.


In [ ]:
EB_PATH = PROJECT_ROOT / "data" / "trajectorynet_eb" / "eb_velocity_v5.npz"
EB_SOURCE_PATH = PROJECT_ROOT.parent / "baselines" / "trajectorynet" / "data" / "eb_velocity_v5.npz"
if not EB_PATH.exists() and EB_SOURCE_PATH.exists():
    copy_trajectorynet_eb_to_data(EB_SOURCE_PATH, EB_PATH)

eb = load_eb_timecourse_for_ch03(EB_PATH, max_cells_per_time=900 if QUICK_MODE else None, seed=SEED)
cost_embedding = "X_cost"
plot_embedding = "X_plot"
n_cost_dims = int(eb[cost_embedding].shape[1])

time_counts = pd.Series(eb["time"].astype(str)).value_counts().sort_index()
time_order = list(time_counts.index)
source_time = time_order[0]
target_time = time_order[1]
mask_source = eb["time"].astype(str) == source_time
mask_target = eb["time"].astype(str) == target_time
X_source_20d = np.asarray(eb["X_cost"][mask_source], dtype=np.float32)
X_target_20d = np.asarray(eb["X_cost"][mask_target], dtype=np.float32)
X_source_phate = np.asarray(eb["X_plot"][mask_source], dtype=np.float32)
X_target_phate = np.asarray(eb["X_plot"][mask_target], dtype=np.float32)
count_preview = pd.DataFrame({"time": time_counts.index, "n_cells": time_counts.values})
tracker.display_table(count_preview, n=8)


In [ ]:
src_train_idx, src_val_idx = ch03.train_val_indices(len(X_source_20d), seed=SEED + 11)
tgt_train_idx, tgt_val_idx = ch03.train_val_indices(len(X_target_20d), seed=SEED + 12)
X0_train, X0_val = X_source_20d[src_train_idx], X_source_20d[src_val_idx]
X1_train, X1_val = X_target_20d[tgt_train_idx], X_target_20d[tgt_val_idx]
X0_train_phate, X0_val_phate = X_source_phate[src_train_idx], X_source_phate[src_val_idx]
X1_train_phate, X1_val_phate = X_target_phate[tgt_train_idx], X_target_phate[tgt_val_idx]

pair_batch_fn = ch03.make_random_pair_batch_fn(X0_train, X1_train, seed=SEED + 13)

val_pair_n = 1200 if QUICK_MODE else 2500
if SMOKE_MODE:
    val_pair_n = 180
val_pair_rng = np.random.default_rng(SEED + 13)
val_i0 = val_pair_rng.integers(0, len(X0_val), size=int(val_pair_n))
val_i1 = val_pair_rng.integers(0, len(X1_val), size=int(val_pair_n))
val_x0 = X0_val[val_i0].astype(np.float32)
val_x1 = X1_val[val_i1].astype(np.float32)
val_t_grid = np.asarray([0.25, 0.50, 0.75], dtype=np.float32)
print({"X0_train": X0_train.shape, "X1_train": X1_train.shape, "X0_val": X0_val.shape, "X1_val": X1_val.shape})


In [ ]:
MAIN_MODEL_PATH = OUT_DIR / f"ch03_eb20d_velocity_mlp_seed{SEED}.pt"
MAIN_CONFIG_PATH = OUT_DIR / f"ch03_eb20d_main_config_seed{SEED}.json"
if not MAIN_MODEL_PATH.exists() or not MAIN_CONFIG_PATH.exists():
    raise FileNotFoundError(
        "Missing EB main model cache. Run notebooks/chapter3_2_eb_flow_matching.ipynb before this baseline notebook."
    )
main_model_config = json.loads(MAIN_CONFIG_PATH.read_text())
eb_model = VelocityMLP(
    x_dim=int(main_model_config.get("x_dim", 20)),
    hidden_dim=int(main_model_config.get("hidden_dim", 256)),
    hidden_layers=int(main_model_config.get("hidden_layers", 4)),
).to(DEVICE)
eb_model.load_state_dict(torch.load(MAIN_MODEL_PATH, map_location=DEVICE))
eb_model.eval()
print({"loaded_model_cache": str(MAIN_MODEL_PATH.relative_to(PROJECT_ROOT)), "main_steps": main_model_config.get("steps")})


In [ ]:
from sklearn.neighbors import KNeighborsRegressor

knn_neighbors = min(20, len(eb["X_cost"]))
phate_projector = KNeighborsRegressor(n_neighbors=knn_neighbors, weights="distance")
phate_projector.fit(np.asarray(eb["X_cost"], dtype=np.float32), np.asarray(eb["X_plot"], dtype=np.float32))

sample_n = min(1200 if QUICK_MODE else 2200, len(X0_val))
if SMOKE_MODE:
    sample_n = min(150, len(X0_val))
sample_idx = subsample_idx(len(X0_val), sample_n, seed=SEED + 31)
x0_eval_20d = torch.as_tensor(X0_val[sample_idx], dtype=torch.float32, device=DEVICE)
target_eval_20d = X1_val.astype(np.float32)
print({"eval_source_cells": int(sample_n), "target_eval_cells": int(len(target_eval_20d)), "projection": "kNN X_cost to X_plot for display only"})


## Shared evaluation settings

Repeated metric calls use the same sliced-Wasserstein projection budget. The training and sampling mechanics stay visible inside each ablation cell; shared source helpers handle only generic endpoint metrics, ODE wrappers, time sampling, and artifact writing.


In [ ]:
metric_n_projections = 48 if SMOKE_MODE else 128


## 8. Solver Comparison Lite

This diagnostic asks how much numerical integration choices change endpoint metrics after the EB 20D CFM vector field has already been trained. The configuration fixes the model and held-out target set, varies only sampler and numerical budget, and reports endpoint discrepancy in the 20D PC space. This supports a solver diagnostic for a fixed trained CFM field; it is not a trained CNF baseline and it does not use PHATE coordinates for metrics.


### Hypothesis
This diagnostic tests whether endpoint readouts for a fixed trained EB CFM field are sensitive to the numerical solver family and integration budget.

### Setup
The solver grid is represented by `solver_specs`/`rtols` and documents the Euler/Heun/RK4 sweep alongside the notebook's adaptive ODE settings. The following cells call `euler_sample`, `midpoint_sample`, and `odeint_sample`, then report endpoint MMD through `mmd_20d` and compute cost through NFE.


In [ ]:
solver_specs = [("euler", K) for K in [5, 10, 20, 50, 100]] + [("midpoint", K) for K in [5, 10, 20, 50]]
if SMOKE_MODE:
    solver_specs = [("euler", K) for K in [5, 10]] + [("midpoint", K) for K in [5, 10]]

rtols = [1e-3, 1e-5]
if SMOKE_MODE:
    rtols = [1e-3]

solver_config = pd.DataFrame(
    [{"sampler": name, "steps": steps, "rtol": np.nan, "model": "cached EB CFM"} for name, steps in solver_specs]
    + [{"sampler": "dopri5", "steps": "adaptive", "rtol": rtol, "model": "cached EB CFM"} for rtol in rtols]
)
print(f"solver settings: {len(solver_specs)} fixed-step runs; dopri5 rtols={rtols}; model=cached EB CFM")


In [ ]:
solver_rows = []
for solver_name, K in solver_specs:
    tic = time.perf_counter()
    if solver_name == "euler":
        endpoint_t, traj_t, nfe = euler_sample(eb_model, x0_eval_20d, n_steps=K, return_traj=True)
    elif solver_name == "midpoint":
        endpoint_t, traj_t, nfe = midpoint_sample(eb_model, x0_eval_20d, n_steps=K, return_traj=True)
    else:
        raise ValueError(solver_name)
    wall = time.perf_counter() - tic
    endpoint_20d = as_np(endpoint_t)
    traj_20d = as_np(traj_t)
    solver_rows.append({
        "sampler": solver_name,
        "steps": int(K),
        "rtol": np.nan,
        "nfe": int(nfe),
        "wall_time_sec": float(wall),
        "mmd_20d": float(mmd_rbf(endpoint_20d, target_eval_20d)),
        "sliced_w2_20d": float(sliced_wasserstein_distance(endpoint_20d, target_eval_20d, n_projections=128 if not SMOKE_MODE else 48, seed=SEED + K + (0 if solver_name == "euler" else 1000))),
        "trajectory_straightness_20d": float(trajectory_straightness(traj_20d)),
    })


In [ ]:
for rtol in rtols:
    try:
        tic = time.perf_counter()
        endpoint_t, traj_t, nfe = odeint_sample(eb_model, x0_eval_20d, rtol=rtol, atol=rtol, method="dopri5")
        wall = time.perf_counter() - tic
        endpoint_20d = as_np(endpoint_t)
        traj_20d = as_np(traj_t)
        solver_rows.append({
            "sampler": "dopri5",
            "steps": "adaptive",
            "rtol": float(rtol),
            "nfe": int(nfe),
            "wall_time_sec": float(wall),
            "mmd_20d": float(mmd_rbf(endpoint_20d, target_eval_20d)),
            "sliced_w2_20d": float(sliced_wasserstein_distance(endpoint_20d, target_eval_20d, n_projections=128 if not SMOKE_MODE else 48, seed=SEED + int(-np.log10(rtol)))),
            "trajectory_straightness_20d": float(trajectory_straightness(traj_20d)),
        })
    except ImportError as exc:
        message = f"torchdiffeq unavailable; skipped dopri5 rtol={rtol}: {exc}"
        warnings.warn(message)
        skipped_items.append(message)
    except Exception as exc:
        message = f"dopri5 failed for rtol={rtol}: {type(exc).__name__}: {exc}"
        warnings.warn(message)
        skipped_items.append(message)

solver_table = pd.DataFrame(solver_rows)
tracker.save_csv(solver_table, "table03_01_solver_diagnostics.csv")
solver_table


In [ ]:
tracker.display_table(pd.read_csv(TABLE_DIR / "table03_01_solver_diagnostics.csv"), n=10)


In [ ]:
fig, ax = ch03.plot_nfe_vs_error(solver_table, y="mmd_20d")
ax.set_ylabel("MMD to target in 20D PC space")
ax.set_title("NFE versus endpoint error")
fig03_10_path = save_and_show(fig, "fig03_10_nfe_vs_endpoint_error.png", width=650)


## 9. ODE-in-the-loop Training Cost: FM versus CNF-style Neural ODE

Both methods share one fixed Sinkhorn-OT endpoint-pair pool, the same `VelocityMLP(20, 256, 4)`, the same Adam optimizer, and the same held-out evaluation set.
The only difference is the training objective: FM/CFM performs simulation-free local velocity regression with 1 NFE per update and no ODE in the loop.
The CNF-style baseline integrates an ODE with adjoint backpropagation at every update and minimizes endpoint MSE after the solve.
The question is which method reaches a given endpoint quality per unit of training compute.
We therefore report a compute–quality Pareto curve over training time and endpoint MMD, plus validation velocity MSE in the shared 20D PC space.
Under the finite budgets used here, FM dominates this frontier: it reaches substantially lower endpoint MMD at ~13× lower wall-clock cost, and the CNF-style baseline does not catch up within the same update budget.
We report this as a finite-budget diagnostic on the EB 20D endpoint-pair task, not a universal claim about CNF training.
PHATE-projected endpoint samples are included only as a qualitative display of where the final generated endpoints land relative to held-out target cells.


### Hypothesis
This experiment tests likelihood-free regression vs ODE-in-loop training when the endpoint-pair pool, network class, optimizer, and evaluation set are held fixed.

### Setup
The benchmark is configured by `cost_steps`, `cost_batch_size`, `cost_cnf_rtols`, and `fixed_pair_batch`. `run_compute_quality_benchmark` trains FM with `cfm_loss_from_pairs` or the CNF-style endpoint objective, then compares endpoint MMD, validation velocity MSE, wall time, and NFE/update.


In [ ]:
cnf_endpoint_completed = False
cnf_endpoint_skipped_reason = None
cost_steps = 600 if QUICK_MODE else 3000
cost_batch_size = 64 if QUICK_MODE else 128
cost_cnf_rtols = [1e-3] if QUICK_MODE else [1e-3, 1e-4]
cost_log_every = 50 if QUICK_MODE else 250
cost_eval_every = 100 if QUICK_MODE else 500
cost_sample_steps = 30 if QUICK_MODE else 50
if SMOKE_MODE:
    cost_steps = 8
    cost_batch_size = 32
    cost_cnf_rtols = [2e-3]
    cost_log_every = 4
    cost_eval_every = 8
    cost_sample_steps = 8

cost_eval_n = min(360 if QUICK_MODE else 900, len(X0_val), len(X1_val))
if SMOKE_MODE:
    cost_eval_n = min(90, len(X0_val), len(X1_val))

eval_idx0 = subsample_idx(len(X0_val), cost_eval_n, seed=SEED + 200)
eval_target_idx = subsample_idx(len(X1_val), cost_eval_n, seed=SEED + 201)
e1_eval_x0_20d = X0_val[eval_idx0].astype(np.float32)
e1_eval_target_20d = X1_val[eval_target_idx].astype(np.float32)

C_ot, ot_scale = fm_ot.compute_cost_matrix(X0_train, X1_train, normalize=True)
ot_epsilon = 0.05
fixed_ot_plan, fixed_ot_info = fm_ot.sinkhorn_plan(C_ot, epsilon=ot_epsilon, return_info=True)
fixed_pair_n = max(cost_batch_size * max(cost_steps, 1), cost_batch_size)
fixed_i0, fixed_i1 = fm_ot.sample_from_plan(fixed_ot_plan, fixed_pair_n, seed=SEED + 211)
fixed_ot_pairs = {
    "x0": X0_train[fixed_i0].astype(np.float32),
    "x1": X1_train[fixed_i1].astype(np.float32),
}

def fixed_pair_batch(step, batch_size):
    start_idx = ((int(step) - 1) * int(batch_size)) % len(fixed_i0)
    take = (np.arange(int(batch_size)) + start_idx) % len(fixed_i0)
    return {"x0": fixed_ot_pairs["x0"][take], "x1": fixed_ot_pairs["x1"][take]}

print(
    "ODE-in-loop cost benchmark: "
    f"steps={cost_steps}; batch_size={cost_batch_size}; "
    f"cnf_rtols={cost_cnf_rtols}; fixed_ot_pairs={len(fixed_i0)}; "
    f"sinkhorn_backend={fixed_ot_info.get('backend')}"
)


In [ ]:
def run_compute_quality_benchmark(method, rtol=None, seed_offset=0):
    set_seed(SEED + 210 + seed_offset)
    model = VelocityMLP(x_dim=20, hidden_dim=256, hidden_layers=4).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    use_ode = method.startswith("CNF")
    if use_ode:
        from torchdiffeq import odeint_adjoint
        t_grid_train = torch.tensor([0.0, 1.0], dtype=torch.float32, device=DEVICE)
        ode_func = ch03.EndpointODEFunc(model)
    rows, endpoint_samples, start, cum_nfe = [], {}, time.perf_counter(), 0
    for step in range(1, cost_steps + 1):
        batch = fixed_pair_batch(step, cost_batch_size)
        x0 = torch.as_tensor(batch["x0"], dtype=torch.float32, device=DEVICE)
        x1 = torch.as_tensor(batch["x1"], dtype=torch.float32, device=DEVICE)
        if use_ode:
            ode_func.reset_nfe()
            pred = odeint_adjoint(ode_func, x0, t_grid_train, rtol=float(rtol),
                                  atol=float(rtol), method="dopri5",
                                  adjoint_params=tuple(model.parameters()))
            loss = ((pred[-1] - x1) ** 2).mean(dim=-1).mean()
            step_nfe = int(ode_func.nfe)
        else:
            loss = cfm_loss_from_pairs(model, x0, x1)
            step_nfe = 1
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        cum_nfe += step_nfe
        if step == 1 or step % cost_log_every == 0 or step == cost_steps:
            val_mse = ch03.val_cfm_mse(model, val_x0, val_x1, val_t_grid,
                                       device=DEVICE,
                                       max_eval_pairs=900 if not SMOKE_MODE else 120,
                                       seed=SEED + 14)
            rows.append({"method": method, "rtol": float(rtol) if use_ode else np.nan,
                         "step": int(step), "wall_time_sec": float(time.perf_counter() - start),
                         "cumulative_nfe": int(cum_nfe),
                         "train_nfe_per_update": float(step_nfe),
                         "train_loss_20d": float(loss.detach().cpu()),
                         "val_mse_20d": float(val_mse),
                         "endpoint_mmd_20d": np.nan, "sliced_w2_20d": np.nan})
        if step == 1 or step % cost_eval_every == 0 or step == cost_steps:
            if use_ode:
                ep, nfe_s, met = ch03.eval_ode_endpoint_20d(
                    model, e1_eval_x0_20d, e1_eval_target_20d, rtol=float(rtol),
                    seed=SEED + 231 + seed_offset, device=DEVICE,
                    n_projections=metric_n_projections)
            else:
                ep, nfe_s, met = ch03.eval_cfm_endpoint_20d(
                    model, e1_eval_x0_20d, e1_eval_target_20d,
                    n_steps=cost_sample_steps, seed=SEED + 230 + seed_offset,
                    device=DEVICE, n_projections=metric_n_projections)
            endpoint_samples[int(step)] = ep
            rows[-1]["endpoint_mmd_20d"] = float(met["endpoint_mmd_20d"])
            rows[-1]["sliced_w2_20d"] = float(met["sliced_w2_20d"])
            rows[-1]["sample_nfe"] = int(nfe_s)
    return model, pd.DataFrame(rows), endpoint_samples



In [ ]:
cfm_e1_model, cfm_e1_history, cfm_e1_endpoints = run_compute_quality_benchmark("FM", seed_offset=0)


In [ ]:
cnf_e1_models, cnf_histories, cnf_e1_endpoints = {}, [], {}
try:
    for ri, cnf_rtol in enumerate(cost_cnf_rtols):
        m, h, ep = run_compute_quality_benchmark(f"CNF ODE rtol={cnf_rtol:g}", rtol=cnf_rtol, seed_offset=10 + ri)
        cnf_e1_models[float(cnf_rtol)] = m
        cnf_histories.append(h)
        cnf_e1_endpoints[float(cnf_rtol)] = ep
    cnf_endpoint_completed = True
except Exception as exc:
    cnf_endpoint_skipped_reason = f"{type(exc).__name__}: {exc}"
    ch03.record_skip(skipped_items, "CNF ODE-in-loop cost benchmark", cnf_endpoint_skipped_reason)
cnf_e1_history = pd.concat(cnf_histories, ignore_index=True) if cnf_histories else pd.DataFrame()
e1_history = pd.concat([cfm_e1_history, cnf_e1_history], ignore_index=True) if len(cnf_e1_history) else cfm_e1_history.copy()


In [ ]:
summary_rows = []
final_eval_rows = e1_history[e1_history["endpoint_mmd_20d"].notna()].copy()
fm_eval_final = final_eval_rows[final_eval_rows["method"].eq("FM")].iloc[-1]
fm_mean_sec_per_update = float(fm_eval_final["wall_time_sec"] / max(cost_steps, 1))

for method, group in final_eval_rows.groupby("method", sort=False):
    final_row = group.iloc[-1]
    uses_ode = method.startswith("CNF")
    mean_sec_per_update = float(final_row["wall_time_sec"] / max(cost_steps, 1))
    summary_rows.append({
        "method": method,
        "train_objective": (
            "endpoint_mse_after_adjoint_ode_solve_fixed_OT_pairs_20D"
            if uses_ode else "local_velocity_regression_fixed_OT_pairs_20D"
        ),
        "train_uses_ode": bool(uses_ode),
        "steps": int(cost_steps),
        "batch_size": int(cost_batch_size),
        "rtol": float(final_row["rtol"]) if uses_ode else np.nan,
        "wall_time_sec": float(final_row["wall_time_sec"]),
        "mean_sec_per_update": mean_sec_per_update,
        "train_nfe_per_update": float(final_row["train_nfe_per_update"]),
        "cumulative_nfe": int(final_row["cumulative_nfe"]),
        "relative_slowdown_vs_fm": mean_sec_per_update / fm_mean_sec_per_update,
        "final_val_mse_20d": float(final_row["val_mse_20d"]),
        "endpoint_mmd_20d": float(final_row["endpoint_mmd_20d"]),
        "sliced_w2_20d": float(final_row["sliced_w2_20d"]),
        "sample_nfe": int(final_row["sample_nfe"]),
    })

E1_table = pd.DataFrame(summary_rows)
tracker.save_csv(E1_table, "tableE1_cfm_vs_cnf_endpoint.csv")
E1_table


In [ ]:
tracker.display_table(pd.read_csv(TABLE_DIR / "tableE1_cfm_vs_cnf_endpoint.csv"), n=8)


In [ ]:
method_colors = {"FM": PAPER_COLORS["generated"], "source": PAPER_COLORS["source"], "target": PAPER_COLORS["target_red"]}
method_label = {"FM": "FM"}
for ri, cnf_rtol in enumerate(cost_cnf_rtols):
    name = f"CNF ODE rtol={cnf_rtol:g}"
    label = f"CNF rtol={cnf_rtol:g}"
    method_colors[name] = PAPER_COLORS["cnf"] if ri == 0 else PAPER_COLORS["dopri5"]
    method_label[name] = label

fig, ax = plt.subplots(figsize=(5.9, 3.5))
pareto = e1_history[e1_history["endpoint_mmd_20d"].notna()].copy()
best_rows = []
for method, group in pareto.groupby("method", sort=False):
    group = group.sort_values("wall_time_sec").copy()
    x = group["wall_time_sec"].clip(lower=1e-6)
    y_raw = group["endpoint_mmd_20d"]
    y_best = y_raw.cummin()
    color = method_colors.get(method, "0.35")
    label = method_label.get(method, method)
    ax.scatter(x, y_raw, s=12, color=color, alpha=0.16, linewidths=0)
    ax.plot(x, y_best, marker="o", linewidth=1.8, markersize=3.6,
            color=color, label=label)
    best_rows.append({"method": method, "time": float(x.iloc[-1]),
                      "best_mmd": float(y_best.iloc[-1]), "color": color,
                      "label": label})
fm_final_time = max(float(E1_table.loc[E1_table["method"].eq("FM"), "wall_time_sec"].iloc[0]), 1e-6)
ax.set_xscale("log")
ax.axvline(fm_final_time, color="0.55", linestyle="--", linewidth=0.8)
ax.text(fm_final_time, 0.98, "matched compute", rotation=90,
        transform=ax.get_xaxis_transform(), ha="right", va="top", fontsize=6, color="0.35")
if best_rows:
    xmin = max(float(pareto["wall_time_sec"].clip(lower=1e-6).min()) * 0.75, 1e-6)
    xmax = max(row["time"] for row in best_rows) * 2.05
    ax.set_xlim(left=xmin, right=xmax)
fm_summary = E1_table[E1_table["method"].eq("FM")].iloc[0]
cnf_summary = E1_table[E1_table["method"].str.startswith("CNF")]
if len(cnf_summary):
    best_cnf = cnf_summary.sort_values("endpoint_mmd_20d").iloc[0]
    slowdown = float(best_cnf["relative_slowdown_vs_fm"])
    ax.text(0.05, 0.62, f"FM reaches lower MMD\nwith {slowdown:.1f}x lower wall-clock cost",
            transform=ax.transAxes, ha="left", va="bottom", fontsize=7.4, color="0.20",
            bbox={"facecolor": "white", "edgecolor": "0.85", "alpha": 0.92, "pad": 2.0})
label_box = {"facecolor": "white", "edgecolor": "none", "alpha": 0.88, "pad": 0.8}
for row in best_rows:
    offset, va = (8, 0), "center"
    if row["method"] == "CNF ODE rtol=0.001":
        offset, va = (9, 10), "bottom"
    elif row["method"] == "CNF ODE rtol=0.0001":
        offset, va = (9, -11), "top"
    elif row["method"] == "FM":
        offset, va = (8, 7), "bottom"
    ax.annotate(row["label"], xy=(row["time"], row["best_mmd"]),
                xytext=offset, textcoords="offset points", ha="left", va=va,
                fontsize=7, color=row["color"], bbox=label_box)
ax.set_xlabel("cumulative training time (s, log scale)")
ax.set_ylabel("best-so-far endpoint MMD in 20D")
ax.set_title("Training time vs endpoint quality")
ax.grid(axis="y", color="0.92", linewidth=0.7)
clean_spines(ax)
ax.text(0.02, 0.97, "solid: best-so-far; dots: raw checkpoints",
        transform=ax.transAxes, ha="left", va="top", fontsize=7, color="0.30",
        bbox={"facecolor": "white", "edgecolor": "0.86", "alpha": 0.90, "pad": 1.8})
fig.tight_layout()
figE1_compute_quality_path = save_and_show(fig, "figE1_fm_vs_cnf_compute_quality.png", width=650)






In [ ]:
fig, ax = plt.subplots(figsize=(5.9, 3.5))
for method, group in e1_history.groupby("method", sort=False):
    color = method_colors.get(method, "0.35") if "method_colors" in globals() else None
    label = method_label.get(method, method) if "method_label" in globals() else method
    ax.plot(group["step"], group["wall_time_sec"], marker="o", linewidth=1.5,
            markersize=3.0, color=color, label=label)
    final = group.iloc[-1]
    ax.text(float(final["step"]), float(final["wall_time_sec"]),
            f" {int(final['cumulative_nfe'])} NFE", va="center", fontsize=7, color=color)
ax.set_xlim(left=0, right=float(e1_history["step"].max()) * 1.14)
ax.set_xlabel("optimization step")
ax.set_ylabel("cumulative training seconds")
ax.set_title("Same update budget, different wall-clock cost")
ax.grid(axis="y", color="0.92", linewidth=0.7)
ax.legend(frameon=False)
clean_spines(ax)
figE1_cumulative_time_path = save_and_show(fig, "figE1_fm_vs_cnf_cumulative_time.png", width=650)


## 10. Time Sampling Strategy Ablation

This ablation asks whether changing the distribution of interpolation times `t` changes short-run CFM training and endpoint quality under the same endpoint-pair task. It keeps model capacity, optimizer, validation pairs, and endpoint evaluation fixed while varying the training-time sampler. The result supports only this finite-budget comparison; it does not establish a universal optimal time distribution or a biological mechanism.


### Hypothesis
This ablation tests whether the interpolation-time sampler is an active training assumption for the same EB endpoint-pair task.

### Setup
The sweep is defined by `time_strategy_specs`, `time_steps`, `time_batch_size`, and `time_sample_steps`. `train_cfm_with_time_strategy` calls `sample_t_torch` and `cfm_loss_from_pairs`, and the summary records validation MSE, endpoint MMD, sliced W2, NFE, and wall time.


In [ ]:
time_sampling_completed = False

time_strategy_specs = [
    ("uniform", {}),
    ("logit_normal_sigma_0.5", {"sigma": 0.5}),
    ("logit_normal_sigma_1.0", {"sigma": 1.0}),
    ("logit_normal_sigma_2.0", {"sigma": 2.0}),
    ("beta_2_2", {"alpha": 2.0, "beta": 2.0}),
    ("beta_0.5_0.5", {"alpha": 0.5, "beta": 0.5}),
    ("cosine", {}),
]
time_steps = 350 if QUICK_MODE else 1200
time_batch_size = 192 if QUICK_MODE else 256
time_log_every = 70 if QUICK_MODE else 150
time_sample_steps = 30 if QUICK_MODE else 50
if SMOKE_MODE:
    time_steps = 8
    time_batch_size = 64
    time_log_every = 4
    time_sample_steps = 8

time_config = pd.DataFrame([
    {"strategy": strategy, "parameters": params, "steps": time_steps, "batch_size": time_batch_size, "sample_steps": time_sample_steps}
    for strategy, params in time_strategy_specs
])
print(f"time-sampling strategies: {list(time_config['strategy'])}; steps={time_steps}; batch_size={time_batch_size}; sample_steps={time_sample_steps}")


In [ ]:
def train_cfm_with_time_strategy(strategy, steps, batch_size, log_every, seed_offset=0):
    set_seed(SEED + 300 + seed_offset)
    model = VelocityMLP(x_dim=20, hidden_dim=256, hidden_layers=4).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    local_pair_batch = ch03.make_random_pair_batch_fn(X0_train, X1_train, seed=SEED + 310 + seed_offset)
    rows = []
    start = time.perf_counter()
    last_time = start
    last_step = 0
    for step in range(1, int(steps) + 1):
        batch = local_pair_batch(batch_size)
        x0 = torch.as_tensor(batch["x0"], dtype=torch.float32, device=DEVICE)
        x1 = torch.as_tensor(batch["x1"], dtype=torch.float32, device=DEVICE)
        t_batch = ch03.sample_t_torch(strategy, batch_size, DEVICE, seed=SEED + 320 + seed_offset * 100000 + step, strategy_specs=dict(time_strategy_specs))
        loss = cfm_loss_from_pairs(model, x0, x1, s=t_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step == 1 or step % int(log_every) == 0 or step == int(steps):
            now = time.perf_counter()
            val_mse = ch03.val_cfm_mse(model, val_x0, val_x1, val_t_grid, device=DEVICE, max_eval_pairs=900 if not SMOKE_MODE else 120, seed=SEED + 14)
            rows.append({"strategy": strategy, "step": int(step), "train_loss_20d": float(loss.detach().cpu()), "val_mse_20d": float(val_mse), "wall_time_sec": float(now - start), "sec_per_step": float((now - last_time) / max(1, step - last_step))})
            last_time = now
            last_step = step
    return model, pd.DataFrame(rows)


In [ ]:
time_histories = []
time_summary_rows = []
for si, (strategy, _) in enumerate(time_strategy_specs):
    model, hist = train_cfm_with_time_strategy(strategy, time_steps, time_batch_size, time_log_every, seed_offset=si)
    endpoint_20d, sample_nfe, met = ch03.eval_cfm_endpoint_20d(model, e1_eval_x0_20d, e1_eval_target_20d, n_steps=time_sample_steps, seed=SEED + 400 + si, device=DEVICE, n_projections=metric_n_projections)
    hist["endpoint_mmd_20d"] = np.nan
    hist.loc[hist.index[-1], "endpoint_mmd_20d"] = met["endpoint_mmd_20d"]
    time_histories.append(hist)
    time_summary_rows.append({
        "strategy": strategy, "steps": int(time_steps),
        "final_train_loss_20d": float(hist["train_loss_20d"].iloc[-1]),
        "final_val_mse_20d": float(hist["val_mse_20d"].iloc[-1]),
        "endpoint_mmd_20d": met["endpoint_mmd_20d"], "sliced_w2_20d": met["sliced_w2_20d"],
        "sample_nfe": int(sample_nfe), "wall_time_sec": float(hist["wall_time_sec"].iloc[-1]),
    })

time_history = pd.concat(time_histories, ignore_index=True)
time_table = pd.DataFrame(time_summary_rows)
tracker.save_csv(time_table, "tableE3_time_sampling_ablation.csv")
time_sampling_completed = True
time_table


In [ ]:
tracker.display_table(pd.read_csv(TABLE_DIR / "tableE3_time_sampling_ablation.csv"), n=8)


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.7))
bins = np.linspace(0.0, 1.0, 55)
for i, (strategy, _) in enumerate(time_strategy_specs):
    samples = ch03.sample_t_numpy(strategy, 8000 if not SMOKE_MODE else 1200, seed=SEED + 330 + i, strategy_specs=dict(time_strategy_specs))
    hist, edges = np.histogram(samples, bins=bins, density=True)
    centers = 0.5 * (edges[:-1] + edges[1:])
    ax.plot(centers, hist, linewidth=1.4, label=strategy)
ax.set_xlabel("sampled interpolation time t")
ax.set_ylabel("empirical density")
ax.set_title("Training-time sampling strategies for CFM")
ax.legend(frameon=False, fontsize=6, ncol=2)
ax.grid(axis="y", color="0.92", linewidth=0.7)
time_dist_path = save_and_show(fig, "figE3_time_sampling_distributions.png", width=650)


In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.8))
for strategy, group in time_history.groupby("strategy", sort=False):
    ax.plot(group["step"], group["val_mse_20d"], linewidth=1.35, label=strategy)
ax.set_xlabel("training step")
ax.set_ylabel("validation CFM MSE in 20D")
ax.set_title("Time sampling ablation: validation velocity-regression MSE")
ax.grid(axis="y", color="0.92", linewidth=0.7)
ax.legend(frameon=False, fontsize=6, ncol=2)
time_val_path = save_and_show(fig, "figE3_time_sampling_val_mse.png", width=650)


In [ ]:
fig, ax = plt.subplots(figsize=(6.6, 3.6))
order = time_table.sort_values("endpoint_mmd_20d")["strategy"].tolist()
sns.barplot(data=time_table, x="strategy", y="endpoint_mmd_20d", order=order, ax=ax, color="#2F6B5E")
ax.set_xlabel("time sampling strategy")
ax.set_ylabel("endpoint MMD in 20D")
ax.set_title("Final endpoint MMD after short CFM runs")
ax.tick_params(axis="x", rotation=35)
ax.grid(axis="y", color="0.92", linewidth=0.7)
time_mmd_path = save_and_show(fig, "figE3_time_sampling_endpoint_mmd.png", width=650)

fig, axes = plt.subplots(1, 2, figsize=(8.8, 3.6))
sns.barplot(data=time_table, x="strategy", y="endpoint_mmd_20d", order=order, ax=axes[0], color="#2F6B5E")
sns.barplot(data=time_table, x="strategy", y="sliced_w2_20d", order=order, ax=axes[1], color="#B9352B")
for ax, ylabel in zip(axes, ["endpoint MMD in 20D", "Sliced W2 in 20D"]):
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=40)
    ax.grid(axis="y", color="0.92", linewidth=0.7)
axes[0].set_title("MMD")
axes[1].set_title("Sliced W2")
fig.suptitle("Time sampling ablation final endpoint metrics")
time_bar_path = save_and_show(fig, "figE3_time_sampling_final_bar.png", width=650)


## 11. Network Capacity Ablation

This ablation asks how sensitive validation velocity MSE and endpoint distribution metrics are to VelocityMLP width/depth on the same EB endpoint-pair task. Capacity changes hidden size and layer count while the optimizer, pair source, validation pairs, and endpoint evaluation stay fixed. The section reports finite-budget behavior for a small capacity grid; it does not prove a globally optimal architecture or that capacity alone explains biological transition quality.


### Hypothesis
This ablation tests whether VelocityMLP width and depth are capacity assumptions for the same EB endpoint-pair supervision problem.

### Setup
The grid is stored in `capacity_configs` with shared `capacity_steps`, `capacity_batch_size`, and `capacity_sample_steps`. Each model uses `cfm_loss_from_pairs`, then `capacity_table` reports parameter count, validation MSE, endpoint MMD, sliced W2, NFE, and wall time.


In [ ]:
capacity_ablation_completed = False
capacity_configs = [
    {"config": "tiny", "hidden_dim": 64, "hidden_layers": 3},
    {"config": "small", "hidden_dim": 128, "hidden_layers": 4},
    {"config": "medium", "hidden_dim": 256, "hidden_layers": 4},
]
if not QUICK_MODE:
    capacity_configs.append({"config": "large", "hidden_dim": 512, "hidden_layers": 6})

capacity_steps = 300 if QUICK_MODE else 1200
capacity_batch_size = 192 if QUICK_MODE else 256
capacity_log_every = 100 if QUICK_MODE else 200
capacity_sample_steps = 30 if QUICK_MODE else 50
if SMOKE_MODE:
    capacity_steps = 8
    capacity_batch_size = 64
    capacity_log_every = 4
    capacity_sample_steps = 8

capacity_config_table = pd.DataFrame([
    {**cfg, "steps": capacity_steps, "batch_size": capacity_batch_size, "sample_steps": capacity_sample_steps}
    for cfg in capacity_configs
])
print(f"capacity configs: {list(capacity_config_table['config'])}; steps={capacity_steps}; batch_size={capacity_batch_size}; sample_steps={capacity_sample_steps}")


In [ ]:
capacity_rows = []
capacity_histories = []
for ci, cfg in enumerate(capacity_configs):
    set_seed(SEED + 500 + ci)
    model = VelocityMLP(x_dim=20, hidden_dim=cfg["hidden_dim"], hidden_layers=cfg["hidden_layers"]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    local_pair_batch = ch03.make_random_pair_batch_fn(X0_train, X1_train, seed=SEED + 510 + ci)
    start = time.perf_counter()
    last_loss = np.nan
    hist_rows = []
    for step in range(1, int(capacity_steps) + 1):
        batch = local_pair_batch(capacity_batch_size)
        x0 = torch.as_tensor(batch["x0"], dtype=torch.float32, device=DEVICE)
        x1 = torch.as_tensor(batch["x1"], dtype=torch.float32, device=DEVICE)
        loss = cfm_loss_from_pairs(model, x0, x1)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        last_loss = float(loss.detach().cpu())
        if step == 1 or step % capacity_log_every == 0 or step == int(capacity_steps):
            val_mse = ch03.val_cfm_mse(model, val_x0, val_x1, val_t_grid, device=DEVICE, max_eval_pairs=900 if not SMOKE_MODE else 120, seed=SEED + 14)
            hist_rows.append({"config": cfg["config"], "step": int(step), "train_loss_20d": last_loss, "val_mse_20d": val_mse})
    wall = time.perf_counter() - start
    endpoint_20d, sample_nfe, met = ch03.eval_cfm_endpoint_20d(model, e1_eval_x0_20d, e1_eval_target_20d, n_steps=capacity_sample_steps, seed=SEED + 540 + ci, device=DEVICE, n_projections=metric_n_projections)
    hist = pd.DataFrame(hist_rows)
    capacity_histories.append(hist)
    capacity_rows.append({
        "config": cfg["config"], "input_dim": 20, "output_dim": 20, "hidden_dim": int(cfg["hidden_dim"]),
        "hidden_layers": int(cfg["hidden_layers"]), "n_parameters": int(count_parameters(model)), "lr": 1e-3,
        "batch_size": int(capacity_batch_size), "steps": int(capacity_steps), "final_train_mse_20d": float(last_loss),
        "final_val_mse_20d": float(hist["val_mse_20d"].iloc[-1]), "endpoint_mmd_20d": met["endpoint_mmd_20d"],
        "sliced_w2_20d": met["sliced_w2_20d"], "sample_nfe": int(sample_nfe), "wall_time_sec": float(wall),
    })

capacity_table = pd.DataFrame(capacity_rows)
capacity_history = pd.concat(capacity_histories, ignore_index=True)
tracker.save_csv(capacity_table, "tableT2_training_hyperparams_capacity.csv")
capacity_ablation_completed = True
capacity_table


In [ ]:
tracker.display_table(pd.read_csv(TABLE_DIR / "tableT2_training_hyperparams_capacity.csv"), n=8)


In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 3.4))
ax.plot(capacity_table["n_parameters"], capacity_table["final_val_mse_20d"], marker="o", linewidth=1.5, color="#2F6B5E")
for _, row in capacity_table.iterrows():
    ax.text(row["n_parameters"], row["final_val_mse_20d"], row["config"], fontsize=7, ha="left", va="bottom")
ax.set_xscale("log")
ax.set_xlabel("trainable parameters")
ax.set_ylabel("final validation MSE in 20D")
ax.set_title("Capacity ablation: validation velocity-regression error")
ax.grid(axis="y", color="0.92", linewidth=0.7)
capacity_val_path = save_and_show(fig, "figE2_capacity_val_mse.png", width=650)

fig, ax = plt.subplots(figsize=(5.6, 3.4))
ax.plot(capacity_table["n_parameters"], capacity_table["endpoint_mmd_20d"], marker="s", linewidth=1.5, color="#B9352B")
for _, row in capacity_table.iterrows():
    ax.text(row["n_parameters"], row["endpoint_mmd_20d"], row["config"], fontsize=7, ha="left", va="bottom")
ax.set_xscale("log")
ax.set_xlabel("trainable parameters")
ax.set_ylabel("endpoint MMD in 20D")
ax.set_title("Capacity ablation: endpoint distribution error")
ax.grid(axis="y", color="0.92", linewidth=0.7)
capacity_mmd_path = save_and_show(fig, "figE2_capacity_endpoint_mmd.png", width=650)


## 12. Trajectory Straightness in 20D

This diagnostic asks how straight the numerical paths generated by the main EB CFM sampler are in the 20D training space. A straightness ratio near 1 means path length is close to endpoint displacement. PHATE appears only after the 20D scores are computed, for representative trajectory display; the result is a numerical trajectory diagnostic, not a lineage certificate.


### Hypothesis
This diagnostic tests whether the cached EB CFM sampler follows nearly straight paths in the 20D training space rather than only matching endpoints.

### Setup
The cell samples validation sources with `straight_sample_n` and rolls out `eb_model` using `euler_sample` over `straight_steps`. `per_trajectory_straightness` and `straight_table` summarize straightness ratios, while PHATE projection is used only for representative trajectory display.


In [ ]:
straightness_completed = False
straight_sample_n = min(900 if QUICK_MODE else 1600, len(X0_val))
straight_steps = 50 if QUICK_MODE else 100
if SMOKE_MODE:
    straight_sample_n = min(120, len(X0_val))
    straight_steps = 10

print(f"straightness diagnostic: sample_n={straight_sample_n}; euler_steps={straight_steps}; metric_space=20D PC X_cost; display_space=PHATE X_plot only")


In [ ]:
straight_idx = subsample_idx(len(X0_val), straight_sample_n, seed=SEED + 600)
straight_x0_t = torch.as_tensor(X0_val[straight_idx], dtype=torch.float32, device=DEVICE)
eb_model.eval()
with torch.no_grad():
    straight_endpoint_t, straight_traj_t, straight_nfe = euler_sample(eb_model, straight_x0_t, n_steps=straight_steps, return_traj=True)
straight_traj_20d = as_np(straight_traj_t)
path_length_20d, endpoint_distance_20d, straight_ratio = ch03.per_trajectory_straightness(straight_traj_20d)
q25, q50, q75, q90 = np.quantile(straight_ratio, [0.25, 0.50, 0.75, 0.90])
quantile_group = np.full(straight_ratio.shape, "middle", dtype=object)
quantile_group[straight_ratio <= q25] = "lower_quantile"
quantile_group[straight_ratio >= q75] = "upper_quantile"
straight_table = pd.DataFrame({
    "cell_index": straight_idx.astype(int),
    "endpoint_distance_20d": endpoint_distance_20d,
    "path_length_20d": path_length_20d,
    "straightness_ratio_20d": straight_ratio,
    "quantile_group": quantile_group,
})
tracker.save_csv(straight_table, "tableE5_trajectory_straightness.csv")
straightness_completed = True
straight_table.describe(include="all")


In [ ]:
tracker.display_table(pd.read_csv(TABLE_DIR / "tableE5_trajectory_straightness.csv"), n=8)


In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 3.4))
ax.hist(straight_ratio, bins=36, color="#2F6B5E", alpha=0.78, edgecolor="white", linewidth=0.4)
for value, label, color in [(straight_ratio.mean(), "mean", "#2F6B5E"), (q50, "median", "0.20"), (q90, "90th pct", "#B9352B")]:
    ax.axvline(value, color=color, linestyle="--", linewidth=1.2)
    ax.text(value, ax.get_ylim()[1] * 0.92, label, rotation=90, va="top", ha="right", fontsize=7, color=color)
ax.set_xlabel("20D straightness ratio")
ax.set_ylabel("trajectory count")
ax.set_title("Trajectory straightness under main EB CFM sampler")
ax.grid(axis="y", color="0.92", linewidth=0.7)
straight_hist_path = save_and_show(fig, "figE5_straightness_hist.png", width=650)

fig, ax = plt.subplots(figsize=(5.2, 3.6))
ax.scatter(endpoint_distance_20d, straight_ratio, s=9, color="#2F6B5E", alpha=0.38, linewidths=0)
ax.set_xlabel("endpoint distance in 20D")
ax.set_ylabel("straightness ratio")
ax.set_title("Endpoint displacement versus 20D path straightness")
ax.grid(axis="y", color="0.92", linewidth=0.7)
straight_scatter_path = save_and_show(fig, "figE5_endpoint_distance_vs_straightness.png", width=650)


In [ ]:
low_candidates = np.flatnonzero(straight_ratio <= q25)
high_candidates = np.flatnonzero(straight_ratio >= q75)
low_show = low_candidates[subsample_idx(len(low_candidates), min(8, len(low_candidates)), seed=SEED + 610)]
high_show = high_candidates[subsample_idx(len(high_candidates), min(8, len(high_candidates)), seed=SEED + 611)]
fig, ax = plt.subplots(figsize=(5.8, 4.6))
tidx = subsample_idx(len(X_target_phate), 700, seed=SEED + 612)
ax.scatter(X_target_phate[tidx, 0], X_target_phate[tidx, 1], s=5, color="0.70", alpha=0.18, linewidths=0, label="real target")
traj_phate_arrays = []
for j in low_show:
    pts = phate_projector.predict(straight_traj_20d[:, j, :])
    traj_phate_arrays.append(pts)
    ax.plot(pts[:, 0], pts[:, 1], color="#2F6B5E", alpha=0.58, linewidth=1.0)
    ax.scatter(pts[[0, -1], 0], pts[[0, -1], 1], s=14, color="#2F6B5E", alpha=0.70, linewidths=0)
for j in high_show:
    pts = phate_projector.predict(straight_traj_20d[:, j, :])
    traj_phate_arrays.append(pts)
    ax.plot(pts[:, 0], pts[:, 1], color="#B9352B", alpha=0.60, linewidth=1.0)
    ax.scatter(pts[[0, -1], 0], pts[[0, -1], 1], s=14, color="#B9352B", alpha=0.72, linewidths=0)
all_traj_plot = np.vstack(traj_phate_arrays) if traj_phate_arrays else X_target_phate[tidx]
xlim, ylim = robust_limits(X_target_phate[tidx], all_traj_plot, margin=0.10)
format_axis(ax, xlim, ylim, xlabel="PHATE 1", ylabel="PHATE 2", title="Representative 20D trajectories projected to PHATE display only")
ax.plot([], [], color="#2F6B5E", label="lower straightness quantile")
ax.plot([], [], color="#B9352B", label="upper straightness quantile")
ax.legend(frameon=False, loc="best")
straight_traj_path = save_and_show(fig, "figE5_representative_trajectories_phate.png", width=650)


## Paper-ready tables

The export pass writes the CSV, Markdown, and TeX versions consumed by the manuscript and previews each table after serialization.


In [ ]:
solver_paper = ch03.format_solver_diagnostics_paper_table(solver_table)
tracker.save_paper_table(solver_paper, "paper_table03_01_solver_diagnostics")
tracker.display_table(pd.read_csv(TABLE_DIR / "paper_table03_01_solver_diagnostics.csv"), n=8)


In [ ]:
E1_paper = E1_table.copy()
E1_paper["Method"] = E1_paper["method"]
E1_paper["Objective"] = E1_paper["train_objective"].str.replace("_", " ")
E1_paper["Uses ODE"] = E1_paper["train_uses_ode"].map({True: "yes", False: "no"})
E1_paper["Steps"] = E1_paper["steps"].astype(int)
E1_paper["Batch"] = E1_paper["batch_size"].astype(int)
E1_paper["Time (s)"] = ch03.round_float(E1_paper["wall_time_sec"], 2)
E1_paper["ms/update"] = ch03.round_float(1000.0 * E1_paper["mean_sec_per_update"], 1)
E1_paper["NFE/update"] = ch03.round_float(E1_paper["train_nfe_per_update"], 1)
E1_paper["Slowdown"] = ch03.round_float(E1_paper["relative_slowdown_vs_fm"], 1)
E1_paper["Val MSE (20D)"] = ch03.round_float(E1_paper["final_val_mse_20d"], 3)
E1_paper["MMD (20D) ↓"] = ch03.round_float(E1_paper["endpoint_mmd_20d"], 4)
E1_paper["Sliced W2 (20D) ↓"] = ch03.round_float(E1_paper["sliced_w2_20d"], 3)
E1_paper["Mode"] = "QUICK compute-quality run" if QUICK_MODE else "full compute-quality run"
E1_paper = E1_paper[["Method", "Objective", "Uses ODE", "Steps", "Batch", "Time (s)",
                     "ms/update", "NFE/update", "Slowdown", "Val MSE (20D)",
                     "MMD (20D) ↓", "Sliced W2 (20D) ↓", "Mode"]]
tracker.save_paper_table(E1_paper, "paper_tableE1_cfm_vs_cnf_endpoint")
tracker.display_table(E1_paper, n=8)


E1 table notes: MMD (20D) and Sliced W2 (20D) are on different scales and are not directly comparable to each other; both are reported as endpoint-distribution diagnostics in the 20D PC training space. ms/update and NFE/update are per-optimizer-step training costs; Slowdown is relative to FM within the same run mode.


In [ ]:
time_paper = time_table.copy()
time_paper["Strategy"] = time_paper["strategy"].map(short_strategy_label)
time_paper["Steps"] = time_paper["steps"].astype(int)
time_paper["Train MSE (20D)"] = ch03.round_float(time_paper["final_train_loss_20d"], 3)
time_paper["Val MSE (20D)"] = ch03.round_float(time_paper["final_val_mse_20d"], 3)
time_paper["MMD (20D) ↓"] = ch03.round_float(time_paper["endpoint_mmd_20d"], 4)
time_paper["Sliced W2 (20D) ↓"] = ch03.round_float(time_paper["sliced_w2_20d"], 3)
time_paper["NFE"] = time_paper["sample_nfe"].astype(int)
time_paper["Time (s)"] = ch03.round_float(time_paper["wall_time_sec"], 2)
time_paper = time_paper[["Strategy", "Steps", "Train MSE (20D)", "Val MSE (20D)", "MMD (20D) ↓", "Sliced W2 (20D) ↓", "NFE", "Time (s)"]]
tracker.save_paper_table(time_paper, "paper_tableE3_time_sampling_ablation")
tracker.display_table(time_paper, n=8)


In [ ]:
cap_paper = capacity_table.copy()
cap_paper["Config"] = cap_paper["config"]
cap_paper["Input dim"] = cap_paper["input_dim"].astype(int)
cap_paper["Output dim"] = cap_paper["output_dim"].astype(int)
cap_paper["Hidden"] = cap_paper["hidden_dim"].astype(int)
cap_paper["Layers"] = cap_paper["hidden_layers"].astype(int)
cap_paper["Params"] = cap_paper["n_parameters"].astype(int)
cap_paper["Steps"] = cap_paper["steps"].astype(int)
cap_paper["Train MSE (20D)"] = ch03.round_float(cap_paper["final_train_mse_20d"], 3)
cap_paper["Val MSE (20D)"] = ch03.round_float(cap_paper["final_val_mse_20d"], 3)
cap_paper["MMD (20D) ↓"] = ch03.round_float(cap_paper["endpoint_mmd_20d"], 4)
cap_paper["Sliced W2 (20D) ↓"] = ch03.round_float(cap_paper["sliced_w2_20d"], 3)
cap_paper["Time (s)"] = ch03.round_float(cap_paper["wall_time_sec"], 2)
cap_paper = cap_paper[["Config", "Input dim", "Output dim", "Hidden", "Layers", "Params", "Steps", "Train MSE (20D)", "Val MSE (20D)", "MMD (20D) ↓", "Sliced W2 (20D) ↓", "Time (s)"]]
tracker.save_paper_table(cap_paper, "paper_tableT2_training_hyperparams_capacity")
tracker.display_table(cap_paper, n=8)


In [ ]:
straight_summary = pd.DataFrame([
    {"Metric": "Straightness ratio (20D)", "count": int(straight_table["straightness_ratio_20d"].count()), "mean": round(float(straight_table["straightness_ratio_20d"].mean()), 3), "median": round(float(straight_table["straightness_ratio_20d"].median()), 3), "std": round(float(straight_table["straightness_ratio_20d"].std()), 3), "q10": round(float(straight_table["straightness_ratio_20d"].quantile(0.10)), 3), "q90": round(float(straight_table["straightness_ratio_20d"].quantile(0.90)), 3), "max": round(float(straight_table["straightness_ratio_20d"].max()), 3)},
    {"Metric": "Endpoint distance (20D)", "count": int(straight_table["endpoint_distance_20d"].count()), "mean": round(float(straight_table["endpoint_distance_20d"].mean()), 3), "median": round(float(straight_table["endpoint_distance_20d"].median()), 3), "std": round(float(straight_table["endpoint_distance_20d"].std()), 3), "q10": round(float(straight_table["endpoint_distance_20d"].quantile(0.10)), 3), "q90": round(float(straight_table["endpoint_distance_20d"].quantile(0.90)), 3), "max": round(float(straight_table["endpoint_distance_20d"].max()), 3)},
])
tracker.save_paper_table(straight_summary, "paper_tableE5_trajectory_straightness_summary")
tracker.display_table(straight_summary, n=8)


## Take-aways

- *Finding 1* — The ablations reuse the Chapter 3.2 EB split, display projector, and cached main model while keeping metrics in the 20D PC training space.
- *Finding 2* — Solver, training-cost, time-sampling, and capacity diagnostics are framed as finite-budget comparisons rather than universal claims.
- *Finding 3* — The trajectory-straightness diagnostic scores numerical paths in 20D before PHATE is used for representative displays, so it is not a lineage certificate.

Next: → ch4_1
